# Python with MySQL 기초

이 노트북은 `PyMySQL`을 사용해 MySQL에 연결하고 SQL을 직접 실행하는 실습입니다. SQLAlchemy는 사용하지 않습니다.

## 1. 패키지 불러오기와 접속 정보 준비

상위 `docker-compose.yml`의 기본 접속 정보는 다음과 같습니다.

- host: `localhost`
- port: `3306`
- database: `examplesdb`
- user: `urstory`
- password: `u1234`

In [ ]:
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": int(os.getenv("DB_PORT", "3306")),
    "database": os.getenv("DB_NAME", "examplesdb"),
    "user": os.getenv("DB_USER", "urstory"),
    "password": os.getenv("DB_PASSWORD", "u1234"),
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
}

DB_CONFIG

## 2. MySQL 연결 확인

`SELECT VERSION()`을 실행해서 MySQL 서버에 연결되는지 확인합니다.

In [ ]:
with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT VERSION() AS version;")
        result = cur.fetchone()

result

## 3. 실습 테이블 만들기

Python 실습 전용 테이블을 만듭니다. 같은 노트북을 여러 번 실행해도 되도록 `IF NOT EXISTS`를 사용합니다.

In [ ]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS python_students (
    student_id INT AUTO_INCREMENT COMMENT '수강생을 식별하는 자동 증가 기본키',
    name VARCHAR(50) NOT NULL COMMENT '수강생 이름',
    email VARCHAR(120) NOT NULL UNIQUE COMMENT '수강생 이메일, 중복 불가',
    score DECIMAL(5, 2) COMMENT 'Python 실습용 점수',
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT '수강생 등록 시각',
    PRIMARY KEY (student_id)
) COMMENT = 'Python 연동 실습용 수강생 정보';
"""

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(create_table_sql)
    conn.commit()

print("python_students 테이블 준비 완료")

## 4. INSERT 실행하기

값을 SQL 문자열에 직접 붙이지 않고 `%s` 자리표시자와 파라미터를 사용합니다.

In [ ]:
students = [
    ("김민준", "py_minjun@example.com", 92.5),
    ("이서연", "py_seoyeon@example.com", 85.0),
    ("박도윤", "py_doyun@example.com", 78.0),
]

insert_sql = """
INSERT INTO python_students (name, email, score)
VALUES (%s, %s, %s)
ON DUPLICATE KEY UPDATE
    name = VALUES(name),
    score = VALUES(score);
"""

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.executemany(insert_sql, students)
    conn.commit()

print("데이터 입력 완료")

## 5. SELECT 결과 가져오기

`fetchall()`로 조회 결과를 가져온 뒤 pandas DataFrame으로 확인합니다.

In [ ]:
select_sql = """
SELECT
    student_id,
    name,
    email,
    score,
    created_at
FROM python_students
ORDER BY student_id;
"""

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(select_sql)
        rows = cur.fetchall()

pd.DataFrame(rows)

## 6. 조건을 사용해 조회하기

In [ ]:
minimum_score = 80

with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT name, email, score
            FROM python_students
            WHERE score >= %s
            ORDER BY score DESC;
            """,
            (minimum_score,),
        )
        rows = cur.fetchall()

pd.DataFrame(rows)

## 7. UPDATE와 DELETE 실행하기

In [ ]:
with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            UPDATE python_students
            SET score = %s
            WHERE email = %s;
            """,
            (95.0, "py_minjun@example.com"),
        )
        cur.execute(
            """
            SELECT student_id, name, email, score
            FROM python_students
            WHERE email = %s;
            """,
            ("py_minjun@example.com",),
        )
        updated_row = cur.fetchone()
    conn.commit()

updated_row

In [ ]:
with pymysql.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT student_id, name, email
            FROM python_students
            WHERE email = %s;
            """,
            ("py_doyun@example.com",),
        )
        deleted_row = cur.fetchone()
        cur.execute(
            """
            DELETE FROM python_students
            WHERE email = %s;
            """,
            ("py_doyun@example.com",),
        )
    conn.commit()

deleted_row

## 8. mysql_db.py 연결 클래스 사용해보기

In [ ]:
from mysql_db import MySQLDB

db = MySQLDB(DB_CONFIG)

with db.get_conn().cursor() as cur:
    cur.execute("SELECT COUNT(*) AS student_count FROM python_students;")
    count_row = cur.fetchone()

count_row

## 정리

- `pymysql.connect()`로 MySQL에 연결합니다.
- `cursor.execute()`로 SQL을 실행합니다.
- 값은 SQL 문자열에 직접 붙이지 않고 파라미터로 전달합니다.
- 조회 결과는 `fetchone()`, `fetchall()`로 가져올 수 있습니다.
- 변경 쿼리 후에는 `commit()`이 필요합니다.